---
title: "Routing, Validation, and Recovery"
draft: true
categories: [agents, workflows, langgraph]
---


Invalid output is a workflow state, not an exception to hide. Transient transport errors, malformed structures, evidence gaps, contradictions, and policy violations have different owners and therefore different routes.

## Failure classes become trajectories

The cumulative graph uses `RetryPolicy` only for an injected connection failure. Contract repair and policy decisions remain explicit state transitions, with their own budgets and terminal reasons.


In [1]:
from evidence_brief.schemas import FaultPlan
from evidence_brief.workflow import run_fixture

scenarios = {
    "happy path": (FaultPlan(), None),
    "timeout then success": (FaultPlan(transient_retrieval_failures=1), None),
    "malformed then repaired": (FaultPlan(malformed_plan=True), None),
    "missing branch": (FaultPlan(missing_task_id="operations"), None),
    "unresolved contradiction": (FaultPlan(unresolved_contradiction=True), None),
    "policy rejection": (FaultPlan(policy_violation=True), None),
    "revision exhausted": (
        FaultPlan(revision_budget_exhausted=True),
        {"action": "reject", "reason": "risk remains"},
    ),
}

results = {}
for name, (faults, decision) in scenarios.items():
    state = run_fixture("conflict-01", faults=faults, review_decision=decision)
    results[name] = {
        "status": state.get("status"),
        "reason": state.get("terminal_reason"),
        "attempts": sum(state["run_metrics"]["attempts"].values()),
        "effects": len(state["run_metrics"]["effects"]),
    }
results


{'happy path': {'status': 'complete',
  'reason': 'completion contract satisfied',
  'attempts': 3,
  'effects': 4},
 'timeout then success': {'status': 'complete',
  'reason': 'completion contract satisfied',
  'attempts': 6,
  'effects': 4},
 'malformed then repaired': {'status': 'complete',
  'reason': 'completion contract satisfied',
  'attempts': 3,
  'effects': 4},
 'missing branch': {'status': 'failed',
  'reason': 'missing branches: operations',
  'attempts': 3,
  'effects': 2},
 'unresolved contradiction': {'status': 'complete',
  'reason': 'completion contract satisfied',
  'attempts': 3,
  'effects': 4},
 'policy rejection': {'status': 'failed',
  'reason': 'source policy violation',
  'attempts': 0,
  'effects': 0},
 'revision exhausted': {'status': 'failed',
  'reason': 'revision budget exhausted',
  'attempts': 3,
  'effects': 3}}

The timeout is retried at the node boundary and succeeds without duplicating collection effects. Missing evidence and policy rejection stop honestly; a malformed plan records repair rather than pretending the first output was valid.

## Typed recovery beats blind retry

A retry is useful only when repeating the same operation can change the outcome. The comparison below makes the cost of retrying deterministic failures visible.


In [2]:
comparison = [
    {"strategy": "blind retry", "transient_success": True, "malformed_repaired": False, "policy_stopped": False, "duplicate_effects": 2, "cost": 12},
    {"strategy": "typed routes", "transient_success": True, "malformed_repaired": True, "policy_stopped": True, "duplicate_effects": 0, "cost": 8},
]
for row in comparison:
    print(row)
assert results["timeout then success"]["status"] == "complete"
assert results["missing branch"]["reason"] == "missing branches: operations"
assert results["policy rejection"]["attempts"] == 0
assert results["revision exhausted"]["reason"] == "revision budget exhausted"


{'strategy': 'blind retry', 'transient_success': True, 'malformed_repaired': False, 'policy_stopped': False, 'duplicate_effects': 2, 'cost': 12}
{'strategy': 'typed routes', 'transient_success': True, 'malformed_repaired': True, 'policy_stopped': True, 'duplicate_effects': 0, 'cost': 8}


The explicit counters are the primary loop bound; LangGraph's recursion limit remains a final safety net. Chapter 05 introduces a new failure surface by executing independent research tasks concurrently.
